Read all tables 

In [0]:
bronze_df = spark.table("workspace.bronze.orders")

silver_df = spark.table("workspace.silver.superstore_orders_silver")

fact_sales = spark.table("workspace.gold.fact_sales")

dim_customer = spark.table("workspace.gold.dim_customer")

dim_product = spark.table("workspace.gold.dim_product")

dim_location = spark.table("workspace.gold.dim_location")

dim_date = spark.table("workspace.gold.dim_date")

Row Count Validation

In [0]:
row_counts = {
    "bronze_df": bronze_df.count(),
    "silver_df": silver_df.count(),
    "fact_sales": fact_sales.count(),
    "dim_customer": dim_customer.count(),
    "dim_product": dim_product.count(),
    "dim_location": dim_location.count(),
    "dim_date": dim_date.count()
}

row_counts_df = spark.createDataFrame(
    [(k, v) for k, v in row_counts.items()],
    ["table_name", "row_count"]
)

display(row_counts_df)

NULL Checks

In [0]:
from pyspark.sql.functions import col

print(
    "Null Customer Keys:",
    fact_sales.filter(col("customer_key").isNull()).count()
)

print(
    "Null Product Keys:",
    fact_sales.filter(col("product_key").isNull()).count()
)

print(
    "Null Location Keys:",
    fact_sales.filter(col("location_key").isNull()).count()
)

print(
    "Null Date Keys:",
    fact_sales.filter(col("date_key").isNull()).count()
)

Business Rules

In [0]:
negative_sales = fact_sales.filter(
    col("sales") < 0
).count()

print(
    "Negative Sales:",
    negative_sales
)

Check for Orphan Customers

In [0]:
orphan_customer = (

    fact_sales

    .join(
        dim_customer,
        "customer_key",
        "left_anti"
    )

)

print(
    "Orphan Customers:",
    orphan_customer.count()
)

**Data Quality Report**

In [0]:
print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

bronze_rows = bronze_df.count()
silver_rows = silver_df.count()
fact_rows = fact_sales.count()

print(f"Bronze Rows          : {bronze_rows}")
print(f"Silver Rows          : {silver_rows}")
print(f"Fact Rows            : {fact_rows}")
print(f"Negative Sales       : {negative_sales}")

invalid_quantity = fact_sales.filter(col("quantity") <= 0).count()
print(f"Invalid Quantity     : {invalid_quantity}")

print("=" * 50)

if (
    bronze_rows == silver_rows == fact_rows
    and negative_sales == 0
    and invalid_quantity == 0
):
    print("STATUS : PASSED ✅")
else:
    print("STATUS : FAILED ❌")

print("=" * 50)